[![Roboflow Notebooks](https://media.roboflow.com/notebooks/template/bannertest2-2.png?ik-sdk-version=javascript-1.4.3&updatedAt=1672932710194)](https://github.com/roboflow/notebooks)

# How to Train a Re-ID Model for Tracking

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roboflow/re-ID/blob/main/docs/cookbooks/how-to-train-reid-model-for-tracking.ipynb)

Trackers can use a Re-ID model to re-identify objects that were occluded or left the frame, so they keep their ID. To do it accurately, the model has to be trained for your domain. In this notebook you'll learn how to fine-tune a pretrained Re-ID model on SportsMOT, score it the way a tracker uses it, and track with it. The steps are the same for any tracking dataset in the MOT format.

> **Runtime:** T4 GPU.

## Setup

### Check GPU availability

Training runs on every crop many times, so use a GPU runtime. If the cell below fails, go to `Runtime` → `Change runtime type` and pick a GPU.

In [ ]:
!nvidia-smi

### Install `reid`, `trackers` and `rfdetr`

We use `trackers` to download SportsMOT and to track, and `rfdetr` to detect the players we track.

In [ ]:
!pip install -q "reid @ git+https://github.com/roboflow/re-ID@feat/training-plots" trackers rfdetr

### Imports

In [ ]:
from pathlib import Path

import cv2
import numpy as np
import supervision as sv
from IPython import display
from PIL import Image
from rfdetr import RFDETRMedium
from trackers import BoTSORTTracker, ReIDConfig
from trackers.eval import evaluate_mot_sequences

from reid import ReassociationProtocol, ReIDModel, load_identity_folders, plot_training_history
from reid.data.mot import generate_mot_patches

## Download SportsMOT

SportsMOT holds basketball, football and volleyball broadcasts, with a box and an identity for every player in every frame. Each sequence is a clip of a longer video and is named after it, so `v_00HRwkvvjtQ_c001` and `v_00HRwkvvjtQ_c003` are two clips of the same game. Train and val come from different videos. The test split has no annotations, so we download train and val, about 13 GB of frames.

In [ ]:
!trackers download --name sportsmot --split train,val --asset annotations,frames

## Convert tracking data into a Re-ID dataset

To train a Re-ID model we need crops of each identity, and we get them with `generate_mot_patches`, which crops every ground-truth box into one folder per track. To read the folders back with one identity per track, we use `load_identity_folders`.

We train on all of `sportsmot/train` and split the videos of `sportsmot/val` in two. `val/` is the validation set, which `train` scores after every epoch to keep the best one, and `test/` is for scoring the model at the end. We split by video, not by clip, because the clips of one video show the same players under different IDs, so a split by clip would test the model on players it already saw in validation.

In [ ]:
# Split the val videos in two, alternating in name order, and keep the clips of each half
all_clips = sorted(path.name for path in Path("sportsmot/val").iterdir() if (path / "gt").is_dir())
videos = sorted({clip.rsplit("_c", 1)[0] for clip in all_clips})
val_videos = videos[0::2]
val_clips = [clip for clip in all_clips if clip.rsplit("_c", 1)[0] in val_videos]
test_clips = [clip for clip in all_clips if clip.rsplit("_c", 1)[0] not in val_videos]

# Crop each set once, so running the cell again reuses the folders
cuts = [("sportsmot/train", "train", None), ("sportsmot/val", "val", val_clips), ("sportsmot/val", "test", test_clips)]
for mot_dir, crops_dir, clips in cuts:
    if not Path(crops_dir).is_dir():
        generate_mot_patches(mot_dir, crops_dir, split="full", sequences=clips)

train_crops = load_identity_folders("train")
val_crops = load_identity_folders("val")
test_crops = load_identity_folders("test")

for name, crops in (("train", train_crops), ("val", val_crops), ("test", test_crops)):
    print(f"{name}: {len(crops.image_paths)} crops of {len(np.unique(crops.pids))} players")

## Score the model before training

We start from `osnet_x1_0_msmt17_combineall`, a general model pretrained on pedestrians from MSMT17, and score it on `test/` the way a tracker uses it. For that we use `ReassociationProtocol`, which takes each box and ranks the boxes of the same clip 30 frames later by how close their embeddings are. `rank1` is the percentage of boxes whose closest match is the right player, and `mrr` averages 1 / rank of the right one.

In [ ]:
model = ReIDModel.from_pretrained("osnet_x1_0_msmt17_combineall")
protocol = ReassociationProtocol()

before = model.evaluate(test_crops, protocol=protocol, verbose=False)
print(f"rank1 {before.metrics['rank1']:.1f}  mrr {before.metrics['mrr']:.1f}")

## Train

Tracking data needs three changes to the default training.

**Shorter epochs.** Every player is cropped in every frame, so `train/` holds over 300,000 crops, and neighboring frames give nearly the same image. To validate the model more often, we set `steps_per_epoch=1000`, so each epoch draws 1000 batches instead of passing over every crop, and the 20 epochs see each crop about twice.

**Batches from one clip.** The triplet loss learns to separate each player from the others in the batch. To make those others the players a tracker can mix up, the ones in the same scene, we set `sample_per_sequence=True`, which draws each batch from one clip. A player seen in two clips of one game, under two IDs, then never shares a batch with themself.

**Validation like a tracker.** To score the model on `val/` the way we scored it on `test/`, we pass `val_protocol=ReassociationProtocol()`. Before training and after every epoch, `train` scores the model and saves the epoch with the highest MRR to `best/` in `output_dir`.

After every epoch, `train` also plots the embeddings of 10 validation players, under that epoch's line. To load and augment the crops in 2 background processes, we set `num_workers=2`.

In [ ]:
history = model.train(
    train_crops,
    output_dir="runs/sportsmot",
    val_data=val_crops,
    val_protocol=ReassociationProtocol(),
    epochs=20,
    steps_per_epoch=1000,
    sample_per_sequence=True,
    num_workers=2,
)

## Plot the training curves

To plot the losses, the validation scores and the learning rate by epoch, we use `plot_training_history`.

In [ ]:
fig = plot_training_history("runs/sportsmot")

Validation MRR rises fast once the backbone starts training and then slowly, so the model learns most of what tells the players apart in the first epochs. It also moves up and down by a point or two between epochs, which is why we keep the best epoch, marked by the dashed line.

The run folder holds `last/`, the model after the last epoch, and `best/`, the epoch with the highest MRR on `val/`. Both load with `from_pretrained` like any model. Next to them, `metrics.csv` holds the rows of `history`, `training_config.json` the run's settings, and `embeddings/` one plot per epoch.

In [ ]:
print(sorted(path.name for path in Path("runs/sportsmot").iterdir()))

## Score the model after training

We now compare the last epoch, which `model` holds, and the best epoch (loaded from `best/`), on the test set. The protocol picks the same queries every time, so the three scores compare the models on the same data.

In [ ]:
best_model = ReIDModel.from_pretrained("runs/sportsmot/best")

results = {
    "before training": before,
    "last epoch": model.evaluate(test_crops, protocol=protocol, verbose=False),
    "best epoch": best_model.evaluate(test_crops, protocol=protocol, verbose=False),
}
print(f"{'':<18}{'rank1':>8}{'mrr':>8}")
for name, result in results.items():
    print(f"{name:<18}{result.metrics['rank1']:>8.1f}{result.metrics['mrr']:>8.1f}")

## Track the players

Now we track one test clip, `v_5ekaksddqrc_c002`, with BoT-SORT on RF-DETR detections three times: without Re-ID, with the general model and with the trained one from `best/`.

In [ ]:
clip = "v_5ekaksddqrc_c002"
detector = RFDETRMedium()
general_model = ReIDModel.from_pretrained("osnet_x1_0_msmt17_combineall")
trained_model = ReIDModel.from_pretrained("runs/sportsmot/best")
# The values the trackers Re-ID guide tuned for SportsMOT
reid_config = ReIDConfig(appearance_threshold=0.3425, proximity_threshold=0.9953)
trackers_by_name = {
    "boxes_only": BoTSORTTracker(frame_rate=25),
    "general": BoTSORTTracker(frame_rate=25, reid_model=general_model, reid_config=reid_config),
    "trained": BoTSORTTracker(frame_rate=25, reid_model=trained_model, reid_config=reid_config),
}

# Detect once per frame and feed the same boxes to every tracker
rows = {name: [] for name in trackers_by_name}
frame_paths = sorted(Path(f"sportsmot/val/{clip}/img1").glob("*.jpg"))
for frame_path in frame_paths:
    frame = cv2.imread(str(frame_path))
    detections = detector.predict(frame[:, :, ::-1].copy(), threshold=0.1)  # RF-DETR takes RGB
    detections = detections[detections.class_id == 1]  # people, COCO class 1
    for name, tracker in trackers_by_name.items():
        tracks = tracker.update(detections, frame)
        for (x1, y1, x2, y2), track_id, confidence in zip(tracks.xyxy, tracks.tracker_id, tracks.confidence):
            if track_id >= 0:  # -1 marks a track not confirmed yet
                box = f"{x1:.1f},{y1:.1f},{x2 - x1:.1f},{y2 - y1:.1f}"
                rows[name].append(f"{int(frame_path.stem)},{track_id},{box},{confidence:.3f},-1,-1,-1")

# One MOT-format file per tracker
for name, lines in rows.items():
    Path(f"tracks/{name}").mkdir(parents=True, exist_ok=True)
    Path(f"tracks/{name}/{clip}.txt").write_text("\n".join(lines) + "\n")
print(f"{clip}: {len(frame_paths)} frames")

We score the tracks against SportsMOT's ground truth with `evaluate_mot_sequences` from `trackers`. HOTA weighs finding the players and keeping their identities together, AssA scores only how well the identities hold, and IDF1 is the share of boxes matched to the right player over the whole clip.

In [ ]:
Path("tracks/seqmap.txt").write_text(clip + "\n")
labels = {"boxes_only": "no Re-ID", "general": "general model", "trained": "trained model"}

print(f"{'':<20}{'HOTA':>8}{'IDF1':>8}{'AssA':>8}")
for name, label in labels.items():
    result = evaluate_mot_sequences(
        gt_dir="sportsmot/val",
        tracker_dir=f"tracks/{name}",
        seqmap="tracks/seqmap.txt",
        metrics=["HOTA", "Identity"],
    )
    hota, identity = result.aggregate.HOTA, result.aggregate.Identity
    print(f"{label:<20}{hota.HOTA * 100:>8.1f}{identity.IDF1 * 100:>8.1f}{hota.AssA * 100:>8.1f}")

With the general model, BoT-SORT improves little, from 52.7 to 54.4 HOTA. The trained model raises HOTA to 66.1, IDF1 from 57.6 to 72.2 and AssA from 43.1 to 63.3, so most of the gain comes from keeping identities. Note that this is one clip, so the numbers move more than they would over the whole test set.

In the GIF below, the general model loses the player with the ball as she drives past the defense, and when she reappears it gives her #0 and moves #11 to her teammate. The trained model keeps #11 on her the whole time.

In [ ]:
SWAP_FRAMES = range(196, 257, 2)  # every other frame around the swap
SHOWN_IDS = {"general": (0, 11), "trained": (0, 11)}  # each model's track IDs of the two players

color = sv.ColorPalette.from_hex(
    [
        "#ffff00",
        "#ff9b00",
        "#ff8080",
        "#ff66b2",
        "#ff66ff",
        "#b266ff",
        "#9999ff",
        "#3399ff",
        "#66ffff",
        "#33ff99",
        "#66ff66",
        "#99ff00",
    ]
)
box_annotator = sv.BoxAnnotator(color=color, color_lookup=sv.ColorLookup.TRACK, thickness=3)
label_annotator = sv.LabelAnnotator(
    color=color, color_lookup=sv.ColorLookup.TRACK, text_color=sv.Color.BLACK, text_scale=0.8
)

# Only the two players' tracks, from each model
shown = {}
for name, track_ids in SHOWN_IDS.items():
    rows = np.loadtxt(f"tracks/{name}/{clip}.txt", delimiter=",", ndmin=2)
    shown[name] = rows[np.isin(rows[:, 0], SWAP_FRAMES) & np.isin(rows[:, 1], track_ids)]

# One crop around both players in every frame, so no box or label is cut
height, width = cv2.imread(str(frame_paths[0])).shape[:2]
boxes = np.vstack(list(shown.values()))[:, 2:6]
left, top = np.maximum(boxes[:, :2].min(axis=0) - 60, 0).astype(int)
right, bottom = np.minimum((boxes[:, :2] + boxes[:, 2:4]).max(axis=0) + 60, (width, height)).astype(int)

gif_frames = []
for frame_number in SWAP_FRAMES:
    frame = cv2.imread(f"sportsmot/val/{clip}/img1/{frame_number:06d}.jpg")
    panels = []
    for name, rows in shown.items():
        in_frame = rows[rows[:, 0] == frame_number]
        detections = sv.Detections(
            xyxy=np.column_stack([in_frame[:, 2:4], in_frame[:, 2:4] + in_frame[:, 4:6]]),
            tracker_id=in_frame[:, 1].astype(int),
        )
        track_labels = [f"#{track_id}" for track_id in detections.tracker_id]
        panel = box_annotator.annotate(frame.copy(), detections)
        panel = label_annotator.annotate(panel, detections, labels=track_labels)[top:bottom, left:right]
        title = np.zeros((44, panel.shape[1], 3), dtype=np.uint8)
        cv2.putText(title, labels[name], (12, 32), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
        panels.append(np.vstack([title, panel]))
    gap = np.full((panels[0].shape[0], 8, 3), 255, dtype=np.uint8)
    gif_frames.append(Image.fromarray(cv2.cvtColor(np.hstack([panels[0], gap, panels[1]]), cv2.COLOR_BGR2RGB)))

# Every other frame of 25 frames per second, at half speed, and the middle third, where the IDs change hands, slower
third = len(gif_frames) // 3
durations = [160] * third + [320] * (len(gif_frames) - 2 * third) + [160] * third
gif_frames[0].save("tracking.gif", save_all=True, append_images=gif_frames[1:], duration=durations, loop=0)
display.Image(filename="tracking.gif")

You just fine-tuned a Re-ID model on a tracking dataset and tracked with it. Nice work! For more information see our [documentation guides](https://reid.roboflow.com/guides/training-for-tracking/), which cover more details!

Got feedback or ideas? Open an issue on [GitHub Issues](https://github.com/roboflow/re-ID/issues).